# RAG Data Ingestion Pipeline (MVP)
Building a local vector store (Chroma) for Retrieval-Augmented Generation.

The pipeline loads all PDFs from the `Rag Database` directory, splits them into chunks,
generates embeddings via Google Gemini, and stores everything in a persistent
Chroma DB under `./chroma_db`.

**Dependencies:** `langchain`, `langchain-community`, `langchain-google-genai`,
`langchain-text-splitters`, `pypdf`, `chromadb`, `python-dotenv`

## 1) SETUP – load API key from .env
Same mechanism as in `agent.ipynb` / `app.py`: `GEMINI_API_KEY` is loaded via
`python-dotenv` and validated before use.

In [ ]:
import os
from glob import glob
from dotenv import load_dotenv

# Load API key from .env (same approach as in agent.ipynb / app.py)
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env!")

os.environ["GOOGLE_API_KEY"] = api_key  # for langchain-google-genai
print("Setup complete. GEMINI_API_KEY loaded.")

## 2) DOCUMENT LOADING – dynamically load PDFs from `Rag Database`
All `*.pdf` files in the directory are found automatically and loaded page by page with
`PyPDFLoader`. Each page becomes its own `Document` including metadata
(source path, page number).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

DATA_DIR = "Rag Database"
pdf_files = sorted(glob(os.path.join(DATA_DIR, "*.pdf")))

if not pdf_files:
    raise FileNotFoundError(f"No PDFs found in directory '{DATA_DIR}'!")

documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    documents.extend(loader.load())

print(f"Found PDFs:      {len(pdf_files)}")
for p in pdf_files:
    print(f"  - {p}")
print(f"Loaded pages (documents): {len(documents)}")

## 3) CHUNKING – split text with RecursiveCharacterTextSplitter
The recursive splitter divides the text along natural separators (paragraph, sentence,
word) so that related content stays within a single chunk as much as possible.

- `chunk_size = 1000` (characters per chunk)
- `chunk_overlap = 200` (overlap so context at the edges is preserved)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)
print(f"Created chunks: {len(chunks)}")
print(f"Example metadata of a chunk: {chunks[0].metadata}")

## 4) EMBEDDINGS & VECTOR STORE – create and persist the Chroma database
Embeddings via Google's model `models/gemini-embedding-2`.

**Note on the free-tier quota:** The embedding quota is 100 requests/min.
With a shared/rolled-over API key this window is often briefly exhausted,
which can trigger `429 RESOURCE_EXHAUSTED`. That is why the chunks are inserted
here in batches with retry + backoff – this is more robust than a single
`from_documents` call. The finished DB lives in `./chroma_db` and can later be
loaded without re-embedding.

**Idempotency:** Before inserting, an existing collection is deleted via
`delete_collection()`, so that running the notebook multiple times does not
duplicate the vectors (more robust on Windows than deleting the directory).

In [ ]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

CHROMA_DIR = "./chroma_db"
BATCH_SIZE = 100  # Google's embedding API allows max. 100 texts per request
MAX_TRIES = 6     # retry attempts per batch on 429 (quota exceeded)

def add_with_retry(vs, docs):
    """Inserts a batch; retries on transient errors (429/5xx) with backoff."""
    transient = ("429", "500", "502", "503", "504", "RESOURCE_EXHAUSTED", "Bad Gateway", "getaddrinfo", "ConnectError", "timed out")
    for attempt in range(1, MAX_TRIES + 1):
        try:
            vs.add_documents(docs)
            return
        except Exception as e:
            msg = str(e)
            if any(code in msg for code in transient) and attempt < MAX_TRIES:
                wait = 30 * attempt
                print(f"   Transient error -> waiting {wait}s (attempt {attempt}/{MAX_TRIES - 1})...")
                time.sleep(wait)
            else:
                raise

# Idempotency: delete the existing collection so that running multiple times does
# not create duplicates (via the Chroma API instead of shutil.rmtree = more robust
# on Windows, since Chroma otherwise exclusively locks the index files).
_purge = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)
try:
    _purge.delete_collection()
    print("   Existing collection deleted.")
except Exception:
    print("   No existing collection – fresh build.")

# Create a fresh (persistent) collection and fill it in batches
vectorstore = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)

total = len(chunks)
for start in range(0, total, BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    add_with_retry(vectorstore, batch)
    done = min(start + BATCH_SIZE, total)
    print(f"   Inserted: {done}/{total} chunks")

print(f"\nVector store saved under: {CHROMA_DIR}")
print(f"Indexed vectors: {vectorstore._collection.count()}")

## 5) TEST QUERY – similarity search (Microservices)
An architecture-related test query against the vector store. The top-3 hits are
printed together with their source PDF (metadata).

Note: The DB is already persisted – for later searches it can be loaded without
re-embedding:
```python
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
```

In [ ]:
query = "What are the benefits of a microservices architecture?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "?")
    print(f"--- Result {i} | Source: {source} | Page: {page} ---")
    print(doc.page_content[:300].strip())
    print()